# **Initialize Gemini API**

In [15]:
from google import genai

client = genai.Client()
model = "gemini-3.8-flash"

def read_sas():
    try:
        with open("../data/Project_1/SAS Programs/0 - Data and Measure Standardization_2025Jul.sas", "r", encoding='utf8') as file:
            content = file.read()
            return content
    except FileNotFoundError:
        return f"Error: The file path was not found."

output = read_sas()

# **Get Completion Function**
General purpose function to make an LLM call.

Parameters:
- prompt
    - A user's query
- system_message
    - Provide's the LLM with a role
- model
    - Model to use (e.g. gemini 3.8 flash)
- client
    - gemini, openai, etc
- previous_interaction_id
    - Provide the LLM with context from a pervious call

In [6]:
def get_completion(
        prompt: str, 
        system_message: str, 
        model: str, 
        client,
        previous_interaction_id: str | None = None,
    ) -> tuple[str, str]:
    
    response = client.interactions.create(
        model=model,
        system_instruction=system_message,
        input=prompt,
        previous_interaction_id=previous_interaction_id,
    )

    if not response.output_text:
        raise RuntimeError("Model returned with no text.")

    return response.output_text, response.id

In [7]:
prompt = '''
You are tasked with translating this to spanish: Hello
'''

res, id = get_completion(prompt, "You are a helpful assistant.", model, client)

print(res)

**Hola**


# **Reflection Workflow**

## **Initial Translation**

In [10]:
def initial_translation(
        src_code_language: str,
        trg_code_language: str,
        src_code: str
):
    system_prompt = '''You are an expert in all programming languages. 
        Your task is to translate code from one language to another.
    '''

    translation_prompt = f'''
    Translate the following {src_code_language} code to {trg_code_language}. Preserve all logic, variable names, and 
    functionality. Return only the code block with no extra commentary or details.

    [Source Code]:
    <source_code>
    {src_code}
    <source_code>
    '''

    res, id = get_completion(translation_prompt, system_prompt, model, client)

    return res, id

In [11]:
src_code = '''
def print_hw():
    print("hello, world!")

'''

init_res, init_id = initial_translation("Python", "Java", src_code)

print(init_res)

```java
public class Main {
    public static void print_hw() {
        System.out.println("hello, world!");
    }
}
```


## **Reflection**

## **Improved Translation**